In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_directory = 'rca100/cases/t080'
task_json_path = f'{task_directory}/task.json'

with open(task_json_path, 'r') as file:
    task_data = json.load(file)

alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')

alert_title, alert_window, alert_entity

```
Out[1]:
```
The alert is titled "inventory接口流量下跌告警" (inventory interface traffic drop alert). It occurred between 2026-04-10 18:45:31 and 2026-04-10 18:53:54 (UTC+8). The entity that triggered the alert is an APM operation named "inventory::/api/v1/inventory/{productId}".

The original code execution output of IPython Kernel is also provided below for reference:

('inventory接口流量下跌告警', {'start': '2026-04-10T18:45:31+08:00', 'end': '2026-04-10T18:53:54+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'})```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_path = f'{task_directory}/metrics.parquet'
metrics_data = pd.read_parquet(metrics_path)

# Filter for service-level metrics
service_metrics = metrics_data[metrics_data['entity_set'] == 'apm.service.legacy']

# Ensure 'time' and 'value' columns are numeric
service_metrics['time'] = pd.to_numeric(service_metrics['time'], errors='coerce')
service_metrics['value'] = pd.to_numeric(service_metrics['value'], errors='coerce')

# Aggregate metrics by 'component-KPI' (entity_name + metric)
service_metrics['component_KPI'] = service_metrics['entity_name'] + '-' + service_metrics['metric']
aggregated_metrics = service_metrics.groupby(['component_KPI', 'time'])['value'].sum().reset_index()

# Calculate global thresholds (P95) for each 'component-KPI'
thresholds = aggregated_metrics.groupby('component_KPI')['value'].quantile(0.95).reset_index()
thresholds.columns = ['component_KPI', 'P95']

aggregated_metrics, thresholds

```
Out[2]:
```
The `metrics.parquet` file was successfully loaded, and service-level metrics were aggregated into time series classified by 'component-KPI' (e.g., `accounting-cpu`, `shipping-workload`). Global thresholds (P95) were calculated for each 'component-KPI'. There are 70 unique 'component-KPI' combinations, with their respective P95 values representing the upper threshold for normal behavior.

The original code execution output of IPython Kernel is also provided below for reference:

(          component_KPI              time         value
0        accounting-cpu  1775814331000000      0.007092
1        accounting-cpu  1775814361000000      0.007040
2        accounting-cpu  1775814391000000      0.007054
3        accounting-cpu  1775814421000000      0.007114
4        accounting-cpu  1775814451000000      0.006959
...                 ...               ...           ...
9865  shipping-workload  1775818411000000  10588.000000
9866  shipping-workload  1775818441000000   9925.000000
9867  shipping-workload  1775818471000000  10349.000000
9868  shipping-workload  1775818501000000  10269.000000
9869  shipping-workload  1775818531000000  11084.000000

[9870 rows x 3 columns],               component_KPI           P95
0            accounting-cpu  8.296244e-03
1        accounting-latency  2.839878e-02
2            accounting-mem  9.747388e+07
3       accounting-workload  5.716000e+03
4                    ad-cpu  3.373173e-01
..                      ...           ...
65  recommendation-workload  2.260000e+02
66             shipping-cpu  1.392360e-02
67         shipping-latency  1.374374e-03
68             shipping-mem  6.976853e+06
69        shipping-workload  1.143600e+04

[70 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pytz
from datetime import datetime

# Define the incident time window in UTC+8
timezone = pytz.timezone('Asia/Shanghai')
start_time = int(datetime.fromisoformat(alert_window['start']).astimezone(timezone).timestamp() * 1_000_000)
end_time = int(datetime.fromisoformat(alert_window['end']).astimezone(timezone).timestamp() * 1_000_000)

# Filter the aggregated metrics to include only the time window of the incident
incident_metrics = aggregated_metrics[
    (aggregated_metrics['time'] >= start_time) & (aggregated_metrics['time'] <= end_time)
]

# Merge with thresholds to identify anomalies
incident_metrics = incident_metrics.merge(thresholds, on='component_KPI', how='left')
incident_metrics['is_anomaly'] = incident_metrics['value'] > incident_metrics['P95']

# Filter for anomalies
anomalies = incident_metrics[incident_metrics['is_anomaly']]

incident_metrics, anomalies

```
Out[3]:
```
During the incident time window (2026-04-10 18:45:31 to 2026-04-10 18:53:54 UTC+8), 1190 data points were analyzed across various 'component-KPI' time series. Among them, 104 data points were identified as anomalies because their values exceeded the global thresholds (P95). These anomalies span multiple components and KPIs, indicating potential issues in several areas.

The original code execution output of IPython Kernel is also provided below for reference:

(          component_KPI              time         value           P95  is_anomaly
0        accounting-cpu  1775817931000000      0.007428      0.008296       False
1        accounting-cpu  1775817961000000      0.008087      0.008296       False
2        accounting-cpu  1775817991000000      0.008051      0.008296       False
3        accounting-cpu  1775818021000000      0.008068      0.008296       False
4        accounting-cpu  1775818051000000      0.008131      0.008296       False
...                 ...               ...           ...           ...         ...
1185  shipping-workload  1775818291000000   2422.000000  11436.000000       False
1186  shipping-workload  1775818321000000   7405.000000  11436.000000       False
1187  shipping-workload  1775818351000000   9509.000000  11436.000000       False
1188  shipping-workload  1775818381000000  10426.000000  11436.000000       False
1189  shipping-workload  1775818411000000  10588.000000  11436.000000       False

[1190 rows x 5 columns],                 component_KPI              time         value           P95  is_anomaly
46             accounting-mem  1775818291000000  9.755716e+07  9.747388e+07        True
53        accounting-workload  1775817991000000  5.786000e+03  5.716000e+03        True
68                     ad-cpu  1775817931000000  3.379460e-01  3.373173e-01        True
74                     ad-cpu  1775818111000000  3.376909e-01  3.373173e-01        True
79                     ad-cpu  1775818261000000  3.373266e-01  3.373173e-01        True
...                       ...               ...           ...           ...         ...
1115  recommendation-workload  1775818231000000  2.460000e+02  2.260000e+02        True
1116  recommendation-workload  1775818261000000  2.420000e+02  2.260000e+02        True
1169             shipping-mem  1775818321000000  7.049216e+06  6.976853e+06        True
1172             shipping-mem  1775818411000000  7.034197e+06  6.976853e+06        True
1175        shipping-workload  1775817991000000  1.158100e+04  1.143600e+04        True

[104 rows x 5 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

# Identify consecutive faults in the 'component-KPI' time series
anomalies['time_diff'] = anomalies['time'].diff().fillna(0)
anomalies['is_new_group'] = (anomalies['time_diff'] > 0).astype(int).cumsum()

# Group consecutive anomalies and calculate the extremal breach for each group
grouped_anomalies = anomalies.groupby(['component_KPI', 'is_new_group']).agg(
    start_time=('time', 'min'),
    end_time=('time', 'max'),
    max_breach=('value', 'max'),
    threshold=('P95', 'first')
).reset_index()

# Calculate the breach percentage
grouped_anomalies['breach_percentage'] = (grouped_anomalies['max_breach'] - grouped_anomalies['threshold']) / grouped_anomalies['threshold']

# Filter out isolated noise spikes and minor threshold breaches
significant_faults = grouped_anomalies[
    (grouped_anomalies['end_time'] - grouped_anomalies['start_time'] > 0) &  # Consecutive anomalies
    (grouped_anomalies['breach_percentage'] > 0.5)  # Significant threshold breach
]

significant_faults[['component_KPI', 'start_time', 'end_time', 'breach_percentage']]

```
Out[4]:
```
No significant faults were identified in the 'component-KPI' time series during the incident time window. All anomalies were either isolated noise spikes or had minor threshold breaches (≤ 50% of the extremal).

The original code execution output of IPython Kernel is also provided below for reference:

Empty DataFrame
Columns: [component_KPI, start_time, end_time, breach_percentage]
Index: []```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_path = f'{task_directory}/traces.parquet'
traces_data = pd.read_parquet(traces_path)

# Ensure relevant columns are numeric
traces_data['startTime'] = pd.to_numeric(traces_data['startTime'], errors='coerce')
traces_data['endTime'] = pd.to_numeric(traces_data['endTime'], errors='coerce')
traces_data['duration'] = pd.to_numeric(traces_data['duration'], errors='coerce')

# Filter traces for the 'inventory::/api/v1/inventory/{productId}' operation during the incident time window
filtered_traces = traces_data[
    (traces_data['spanName'] == 'inventory::/api/v1/inventory/{productId}') &
    (traces_data['startTime'] >= start_time * 1_000) &  # Convert microseconds to nanoseconds
    (traces_data['endTime'] <= end_time * 1_000)  # Convert microseconds to nanoseconds
]

# Analyze traces for errors or anomalies
filtered_traces['is_error'] = (filtered_traces['statusCode'] != 0) | (filtered_traces['statusMessage'].notna())
filtered_traces['is_slow'] = filtered_traces['duration'] > filtered_traces['duration'].quantile(0.95)

# Identify root cause by tracing the call chain
error_traces = filtered_traces[filtered_traces['is_error']]
slow_traces = filtered_traces[filtered_traces['is_slow']]

error_traces, slow_traces

```
Out[5]:
```
No traces related to the 'inventory::/api/v1/inventory/{productId}' operation during the incident time window were found to have errors or anomalies, such as high latency or failed requests. Additionally, no issues were identified in the call chain that could indicate a root cause component.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, ip, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, links, is_error, is_slow]
Index: []

[0 rows x 23 columns], Empty DataFrame
Columns: [traceId, spanId, parentSpanId, kind, spanName, startTime, endTime, duration, serviceName, pid, ip, hostname, statusCode, statusMessage, traceState, resources, attributes, __topic__, __tag__:__pack_id__, events, links, is_error, is_slow]
Index: []

[0 rows x 23 columns])```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_path = f'{task_directory}/logs.parquet'
logs_data = pd.read_parquet(logs_path)

# Ensure '_time_' column is numeric
logs_data['_time_'] = pd.to_numeric(logs_data['_time_'], errors='coerce')

# Filter logs for the 'inventory' service during the incident time window
inventory_logs = logs_data[
    (logs_data['_container_name_'].str.contains('inventory', case=False, na=False)) &
    (logs_data['_time_'] >= start_time) &
    (logs_data['_time_'] <= end_time)
]

# Search for error messages, warnings, or other relevant information
error_logs = inventory_logs[inventory_logs['content'].str.contains('error|warn|fail|exception', case=False, na=False)]

inventory_logs, error_logs

```
Out[6]:
```
No logs related to the 'inventory' service were found during the incident time window. Additionally, no error messages, warnings, or relevant information indicating the root cause of the traffic drop were identified in the logs.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns], Empty DataFrame
Columns: [content, _time_, _source_, _container_ip_, _image_name_, _container_name_, _pod_name_, _namespace_, _pod_uid_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns])```
```

In [ ]:
# In[7]:

# Load the topology.json file
topology_path = f'{task_directory}/topology.json'
with open(topology_path, 'r') as file:
    topology_data = json.load(file)

# Extract entities and edges from the topology data
entities = pd.DataFrame(topology_data['entities'])
edges = pd.DataFrame(topology_data['edges'])

# Focus on the 'inventory' service and its operation 'inventory::/api/v1/inventory/{productId}'
inventory_entity = entities[entities['name'].str.contains('inventory', case=False, na=False)]
inventory_edges = edges[
    (edges['src'].isin(inventory_entity['id'])) | (edges['dst'].isin(inventory_entity['id']))
]

inventory_entity, inventory_edges

```
Out[7]:
```
The service dependency graph reveals that the 'inventory' service has several related entities, including operations like 'inventory::/api/v1/inventory/{productId}' and external dependencies such as an HTTP client ('inventory:9090') and a database. The 'inventory' service is also called by another service (ID: 7b50436b7fb9e828f0a8c32a69c53ab0) and makes calls to an external database (ID: d7a7899b0bef364781b3e4cf9cc9b707). These dependencies could be potential sources of the traffic drop issue. However, no direct anomalies or errors were identified in the traces, logs, or metrics for the 'inventory' service or its operation during the incident time window. Further investigation into the upstream and downstream services may be required to identify the root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(                                   id                      type                                               name  first_observed  last_observed                                              props
1    ed13cb14265f98d9eb3b89b664760adb               apm.service                                          inventory               0     1777000616  {'service': 'inventory', 'service_id': '<arms_...
41   474c2072f1451452bbb1abf29cc68b10  apm.external.http_client                                     inventory:9090               0     1775822173                {'instance_addr': 'inventory:9090'}
97   c01a1596a0d45bb7a531f16689910a43             apm.operation  inventory::InventoryMonitoringService.checkAnd...               0     1775822053  {'service': 'inventory', 'operation': 'Invento...
98   0730d69c0818717e0d0e987385f3a3ab             apm.operation                     inventory::/actuator/health/**               0     1775822173  {'service': 'inventory', 'operation': '/actuat...
104  3b0aaa9cc6d1f48a6fce24d7b439f478             apm.operation                        inventory::/actuator/health               0     1775822173  {'service': 'inventory', 'operation': '/actuat...
117  162799adab6799326676266a2c96cf3e             apm.operation           inventory::/api/v1/inventory/{productId}               0     1775822173  {'service': 'inventory', 'operation': '/api/v1...
187  4bae5b3ee7747110acc9ed711c0a46c6                   k8s.pod                         inventory-586695b445-ftpvt      1775805357     1775821785  {'name': 'inventory-586695b445-ftpvt', 'namesp...
189  3ae08cd9ae54be6d06071e007738c321                   k8s.pod                         inventory-586695b445-45vpj      1775805285     1775821785  {'name': 'inventory-586695b445-45vpj', 'namesp...
215  02b4b713642494d79ba63b1f94a57777                   k8s.pod                         inventory-586695b445-pbkvd      1775805321     1775821785  {'name': 'inventory-586695b445-pbkvd', 'namesp...
241  af2679d141fc313291e276608cddbc39               k8s.service                                          inventory      1773319908     1775821785  {'name': 'inventory', 'namespace': 'cms-demo',...
248  33c16dd0c27f7e1a60ad4bc3adcb47ba              apm.instance                                          inventory               0              0  {'service': 'inventory', 'host': '10.0.2.26', ...
251  899ac8edd338f20ba938668e82c3cf36              apm.instance                                          inventory               0              0  {'service': 'inventory', 'host': '10.0.2.172',...
252  23afb9d9bc31156788a5190e42bc1f7f              apm.instance                                          inventory               0              0  {'service': 'inventory', 'host': '10.0.2.123',...
258  829e67af737c07931a908d78b88b9f36                   k8s.pod                         inventory-78697ff45b-7v69l               0              0  {'name': 'inventory-78697ff45b-7v69l', 'namesp...
259  cf3e3b32c756c008d661740d26cf5064                   k8s.pod                         inventory-6d8d574cbb-kmc9p               0              0  {'name': 'inventory-6d8d574cbb-kmc9p', 'namesp...,                                   src       src_type                               dst                  dst_type  relation  first_observed  last_observed
10   469f8e313055adba13ca3f4e76c65505    apm.service  474c2072f1451452bbb1abf29cc68b10  apm.external.http_client     calls               0     1775739373
12   7b50436b7fb9e828f0a8c32a69c53ab0    apm.service  474c2072f1451452bbb1abf29cc68b10  apm.external.http_client     calls               0     1775732953
13   ed13cb14265f98d9eb3b89b664760adb    apm.service  d7a7899b0bef364781b3e4cf9cc9b707     apm.external.database     calls               0     1775739373
28   ed13cb14265f98d9eb3b89b664760adb    apm.service  162799adab6799326676266a2c96cf3e             apm.operation  contains               0     1775739373
31   ed13cb14265f98d9eb3b89b664760adb    apm.service  c01a1596a0d45bb7a531f16689910a43             apm.operation  contains               0     1775739373
56   7b50436b7fb9e828f0a8c32a69c53ab0    apm.service  ed13cb14265f98d9eb3b89b664760adb               apm.service     calls               0     1775732953
63   ed13cb14265f98d9eb3b89b664760adb    apm.service  3b0aaa9cc6d1f48a6fce24d7b439f478             apm.operation  contains               0     1775739373
85   469f8e313055adba13ca3f4e76c65505    apm.service  ed13cb14265f98d9eb3b89b664760adb               apm.service     calls               0     1775739373
92   ed13cb14265f98d9eb3b89b664760adb    apm.service  0730d69c0818717e0d0e987385f3a3ab             apm.operation  contains               0     1775739373
126  0375934342a7f2b01ee80cbbbf203c44  k8s.namespace  af2679d141fc313291e276608cddbc39               k8s.service  contains      1773319908     1775738985
276  ed13cb14265f98d9eb3b89b664760adb    apm.service  33c16dd0c27f7e1a60ad4bc3adcb47ba              apm.instance  contains               0              0
281  ed13cb14265f98d9eb3b89b664760adb    apm.service  899ac8edd338f20ba938668e82c3cf36              apm.instance  contains               0              0
282  ed13cb14265f98d9eb3b89b664760adb    apm.service  23afb9d9bc31156788a5190e42bc1f7f              apm.instance  contains               0              0
293  0375934342a7f2b01ee80cbbbf203c44  k8s.namespace  829e67af737c07931a908d78b88b9f36                   k8s.pod  contains               0              0
294  0375934342a7f2b01ee80cbbbf203c44  k8s.namespace  cf3e3b32c756c008d661740d26cf5064                   k8s.pod  contains               0              0)```
```

In [ ]:
# In[8]:

# Filter metrics for the dependencies of the 'inventory' service during the incident time window
dependency_ids = [
    '474c2072f1451452bbb1abf29cc68b10',  # HTTP client 'inventory:9090'
    'd7a7899b0bef364781b3e4cf9cc9b707',  # Database ID
    '7b50436b7fb9e828f0a8c32a69c53ab0'   # Upstream service ID
]

dependency_metrics = metrics_data[
    (metrics_data['entity_id'].isin(dependency_ids)) &
    (metrics_data['time'] >= start_time) &
    (metrics_data['time'] <= end_time)
]

# Ensure 'time' and 'value' columns are numeric
dependency_metrics['time'] = pd.to_numeric(dependency_metrics['time'], errors='coerce')
dependency_metrics['value'] = pd.to_numeric(dependency_metrics['value'], errors='coerce')

# Merge with thresholds to identify anomalies
dependency_metrics['component_KPI'] = dependency_metrics['entity_name'] + '-' + dependency_metrics['metric']
dependency_metrics = dependency_metrics.merge(thresholds, on='component_KPI', how='left')
dependency_metrics['is_anomaly'] = dependency_metrics['value'] > dependency_metrics['P95']

# Filter for anomalies
dependency_anomalies = dependency_metrics[dependency_metrics['is_anomaly']]

dependency_metrics, dependency_anomalies

```
Out[8]:
```
No metrics data for the dependencies of the 'inventory' service (HTTP client 'inventory:9090', database ID 'd7a7899b0bef364781b3e4cf9cc9b707', and upstream service ID '7b50436b7fb9e828f0a8c32a69c53ab0') was found during the incident time window. Consequently, no anomalies were detected in these dependencies.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service, component_KPI, P95, is_anomaly]
Index: []

[0 rows x 12 columns], Empty DataFrame
Columns: [time, domain, entity_set, entity_id, entity_name, metric, value, metric_set_id, service, component_KPI, P95, is_anomaly]
Index: []

[0 rows x 12 columns])```
```